# 🚀 Notebook 05 — Advanced Patterns for Production

**What you'll learn:**
- Multi-workspace architectures (agent explores multiple data sources)
- Workspace mutations (append, update, delete)
- Custom domain-specific tools alongside built-ins
- Persistent storage with SQLiteStore
- Concurrent access patterns (thread-safe for web servers)
- KV workspaces for config/metadata

## 1. Multi-Workspace Architecture

In [1]:
import json
from ctxtual import Forge, MemoryStore
from ctxtual.utils import paginator, text_search, filter_set, pipeline, kv_reader, text_content

forge = Forge(store=MemoryStore())

# Each data source gets its own workspace type and toolset
users_pager = paginator(forge, "users")
users_search = text_search(forge, "users", fields=["name", "email"])
users_pipe = pipeline(forge, "users")

orders_pager = paginator(forge, "orders")
orders_pipe = pipeline(forge, "orders")

config_kv = kv_reader(forge, "config")

import random
random.seed(42)

@forge.producer(workspace_type="users", toolsets=[users_pager, users_search, users_pipe])
def load_users() -> list[dict]:
    """Load user database."""
    return [
        {"user_id": i, "name": f"User_{i}", "email": f"user{i}@example.com",
         "plan": random.choice(["free", "pro", "enterprise"]),
         "signup_date": f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
         "total_orders": random.randint(0, 100)}
        for i in range(5000)
    ]

@forge.producer(workspace_type="orders", toolsets=[orders_pager, orders_pipe])
def load_orders(user_id: int = None) -> list[dict]:
    """Load orders, optionally filtered by user."""
    orders = [
        {"order_id": i, "user_id": i % 500, "product": f"Product_{random.randint(1, 50)}",
         "amount": round(random.uniform(10, 500), 2),
         "status": random.choice(["pending", "shipped", "delivered"]),
         "date": f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"}
        for i in range(20000)
    ]
    if user_id is not None:
        orders = [o for o in orders if o["user_id"] == user_id]
    return orders

@forge.producer(workspace_type="config", toolsets=[config_kv])
def get_app_config() -> dict:
    """Load application configuration."""
    return {
        "version": "2.4.1",
        "features": {"dark_mode": True, "beta_pipeline": False},
        "limits": {"max_orders_per_user": 100, "rate_limit_rpm": 60},
        "regions": ["us-east-1", "eu-west-1", "ap-southeast-1"],
    }

# Load all three data sources
users_ref = load_users()
orders_ref = load_orders()
config_ref = get_app_config()

print(f"Users:  {users_ref['item_count']:,} items  (tools: {len(users_ref['available_tools'])})")
print(f"Orders: {orders_ref['item_count']:,} items  (tools: {len(orders_ref['available_tools'])})")
print(f"Config: dict workspace         (tools: {len(config_ref['available_tools'])})")

Users:  5,000 items  (tools: 8)
Orders: 20,000 items  (tools: 6)
Config: dict workspace         (tools: 2)


In [2]:
# Cross-workspace query: find top users then look up their orders
# Step 1: Find enterprise users with most orders
top_users = forge.dispatch_tool_call("users_pipe", {
    "workspace_id": users_ref["workspace_id"],
    "steps": [
        {"filter": {"plan": "enterprise"}},
        {"sort": {"field": "total_orders", "order": "desc"}},
        {"limit": 3},
        {"select": ["user_id", "name", "plan", "total_orders"]},
    ],
})
print("Top 3 enterprise users by order count:")
for u in top_users["items"]:
    print(f"  {u['name']} (id={u['user_id']}) — {u['total_orders']} orders")

# Step 2: Check their orders in the orders workspace
for u in top_users["items"][:1]:  # Just first user for demo
    user_orders = forge.dispatch_tool_call("orders_pipe", {
        "workspace_id": orders_ref["workspace_id"],
        "steps": [
            {"filter": {"user_id": u["user_id"]}},
            {"sort": {"field": "amount", "order": "desc"}},
            {"limit": 5},
            {"select": ["order_id", "product", "amount", "status"]},
        ],
    })
    print(f"\nTop 5 orders for {u['name']}:")
    for o in user_orders["items"]:
        print(f"  {o['order_id']}  {o['product']:<15} ${o['amount']:.2f}  {o['status']}")

Top 3 enterprise users by order count:
  User_1288 (id=1288) — 100 orders
  User_1460 (id=1460) — 100 orders
  User_1929 (id=1929) — 100 orders

Top 5 orders for User_1288:


In [3]:
# KV workspace: read configuration
keys = forge.dispatch_tool_call("config_get_keys", {
    "workspace_id": config_ref["workspace_id"],
})
# get_keys has output_hint so it wraps in {"result": ..., "_hint": ...}
keys_data = keys["result"]
print(f"Config keys: {keys_data}")

limits = forge.dispatch_tool_call("config_get_value", {
    "workspace_id": config_ref["workspace_id"],
    "key": "limits",
})
# get_value returns raw value (no hint envelope)
print(f"Limits: {json.dumps(limits, indent=2)}")

Config keys: ['version', 'features', 'limits', 'regions']
Limits: {
  "max_orders_per_user": 100,
  "rate_limit_rpm": 60
}


## 2. Workspace Mutations — Mutable State

In [4]:
# ctxtual supports append, update, patch, and delete operations

@forge.producer(workspace_type="tasks", toolsets=[paginator(forge, "tasks"), pipeline(forge, "tasks")])
def create_task_list() -> list[dict]:
    return [
        {"task_id": 1, "title": "Review PR #42", "status": "open", "priority": "high"},
        {"task_id": 2, "title": "Deploy v2.4.1", "status": "open", "priority": "critical"},
        {"task_id": 3, "title": "Write tests", "status": "in_progress", "priority": "medium"},
    ]

ref = create_task_list()
wid = ref["workspace_id"]
print(f"Initial: {ref['item_count']} tasks")

# Append new items
forge.store.append_items(wid, [
    {"task_id": 4, "title": "Update docs", "status": "open", "priority": "low"},
    {"task_id": 5, "title": "Fix login bug", "status": "open", "priority": "critical"},
])
print(f"After append: {forge.store.count_items(wid)} tasks")

# Patch an item (partial update by index)
forge.store.patch_item(wid, index=2, fields={"status": "done"})  # "Write tests" → done

# Delete items by index — first find done items, then delete by their indices
items = forge.store.get_items(wid)
done_indices = [i for i, item in enumerate(items) if item.get("status") == "done"]
if done_indices:
    forge.store.delete_items(wid, done_indices)
print(f"After deleting done tasks: {forge.store.count_items(wid)} tasks")

# Verify
remaining = forge.dispatch_tool_call("tasks_paginate", {
    "workspace_id": wid, "page": 0, "size": 10,
})
print("\nRemaining tasks:")
for t in remaining["result"]["items"]:
    print(f"  [{t['task_id']}] {t['title']} — {t['status']} ({t['priority']})")

Initial: 3 tasks
After append: 5 tasks
After deleting done tasks: 4 tasks

Remaining tasks:
  [1] Review PR #42 — open (high)
  [2] Deploy v2.4.1 — open (critical)
  [4] Update docs — open (low)
  [5] Fix login bug — open (critical)


## 3. Custom Domain Tools

In [5]:
# Add domain-specific tools alongside built-ins

analytics = forge.toolset("orders")  # Reuse existing toolset!

@analytics.tool(
    name="orders_revenue_by_product",
    output_hint="Use orders_pipe for further analysis on specific products.",
)
def revenue_by_product(workspace_id: str, top_n: int = 10) -> dict:
    """Calculate revenue by product, returning the top N products.

    Args:
        workspace_id: Orders workspace to analyze.
        top_n: Number of top products to return.
    """
    items = analytics.store.get_items(workspace_id)
    revenue = {}
    for item in items:
        p = item.get("product", "unknown")
        revenue[p] = revenue.get(p, 0) + item.get("amount", 0)

    sorted_products = sorted(revenue.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return {
        "products": [{"product": p, "revenue": round(r, 2)} for p, r in sorted_products],
        "total_revenue": round(sum(revenue.values()), 2),
    }

# Use it
result = forge.dispatch_tool_call("orders_revenue_by_product", {
    "workspace_id": orders_ref["workspace_id"],
    "top_n": 5,
})
result_data = result.get("result", result)
print("Top 5 products by revenue:")
for p in result_data["products"]:
    print(f"  {p['product']:<15} ${p['revenue']:>10,.2f}")
print(f"  Total: ${result_data['total_revenue']:>10,.2f}")

Top 5 products by revenue:
  Product_47      $111,934.85
  Product_24      $110,672.26
  Product_35      $110,124.91
  Product_45      $110,065.29
  Product_7       $109,968.44
  Total: $5,123,654.82


## 4. Persistent Storage with SQLiteStore

In [ ]:
import tempfile, os
from ctxtual import Forge, SQLiteStore
from ctxtual.utils import paginator, pipeline

# SQLiteStore persists across process restarts
db_path = os.path.join(tempfile.gettempdir(), "ctx_demo.db")
forge_sql = Forge(store=SQLiteStore(db_path))
sql_pager = paginator(forge_sql, "logs")
sql_pipe = pipeline(forge_sql, "logs")

@forge_sql.producer(workspace_type="logs", toolsets=[sql_pager, sql_pipe])
def ingest_logs() -> list[dict]:
    return [
        {"timestamp": f"2024-01-{(i % 28) + 1:02d}T{i % 24:02d}:00:00",
         "level": random.choice(["INFO", "WARN", "ERROR", "DEBUG"]),
         "service": random.choice(["api", "worker", "scheduler", "db"]),
         "message": f"Event {i}: {random.choice(['request handled', 'timeout', 'connection lost', 'retry succeeded'])}"}
        for i in range(5000)
    ]

ref = ingest_logs()
print(f"Stored {ref['item_count']} logs in SQLite at {db_path}")
print(f"File size: {os.path.getsize(db_path) / 1024:.1f} KB")

# Queries run at SQL level — fast even for large datasets
errors = forge_sql.dispatch_tool_call("logs_pipe", {
    "workspace_id": ref["workspace_id"],
    "steps": [
        {"filter": {"level": "ERROR"}},
        {"count": True},
    ],
})
print(f"ERROR logs: {errors['count']} out of {ref['item_count']}")

# Cleanup
os.unlink(db_path)

## 5. Thread-Safe Concurrent Access

In [ ]:
import threading
from ctxtual import Forge, MemoryStore
from ctxtual.utils import paginator

# Forge is thread-safe — safe for FastAPI / Django / Flask
forge_mt = Forge(store=MemoryStore())
mt_pager = paginator(forge_mt, "data")

@forge_mt.producer(workspace_type="data", toolsets=[mt_pager])
def produce_data(batch_id: int) -> list[dict]:
    return [{"batch": batch_id, "item": i} for i in range(100)]

# Simulate concurrent requests
results = {}
errors = []

def worker(batch_id):
    try:
        ref = produce_data(batch_id)
        page = forge_mt.dispatch_tool_call("data_paginate", {
            "workspace_id": ref["workspace_id"],
            "page": 0, "size": 5,
        })
        results[batch_id] = page["result"]["total"]
    except Exception as e:
        errors.append((batch_id, str(e)))

threads = [threading.Thread(target=worker, args=(i,)) for i in range(20)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Completed {len(results)} concurrent requests, {len(errors)} errors")
assert len(errors) == 0, f"Race conditions detected: {errors}"
print("✅ All requests completed without race conditions")

## 6. Architecture Overview

In [ ]:
# Architecture for a production agent:
#
# ┌──────────────┐   ┌──────────────┐   ┌───────────────┐
# │ Web Scraper  │   │   Database   │   │  API Client   │
# │  (text)      │   │   (list)     │   │   (dict)      │
# └──────┬───────┘   └──────┬───────┘   └──────┬────────┘
#        │                  │                  │
#        ▼                  ▼                  ▼
#   text_content        paginator            kv_reader
#   + search_in_text    + text_search        + get_keys
#   + read_page         + filter_set         + get_value
#                       + pipeline
#        │                   │                   │
#        └───────────────────┼───────────────────┘
#                            ▼
#                    ┌──────────────┐
#                    │    Forge     │
#                    │  (dispatch)  │
#                    └──────┬───────┘
#                           │
#                    ┌──────┴───────┐
#                    │  Agent Loop  │
#                    │  (any LLM)   │
#                    └──────────────┘

print("Architecture supports any combination of data sources and tools.")
print(f"\nThis forge has:")
print(f"  {len(forge.get_all_tool_schemas())} registered tools")
print(f"  System prompt: {len(forge.system_prompt())} chars")
print(f"\nReady for production deployment with any agent framework.")

## Summary of Advanced Patterns

| Pattern | Use Case | Key Feature |
|---------|----------|-------------|
| Multi-workspace | Agent needs multiple data sources | Each source has own type + toolset |
| Mutations | Agent modifies data (task lists, annotations) | `append_items`, `patch_item`, `delete_items` |
| Custom tools | Domain-specific analytics | Add to existing toolset with `@ts.tool()` |
| SQLiteStore | Persistence across restarts | Drop-in replacement for MemoryStore |
| Thread-safety | Web servers (FastAPI, Django) | RLock on Forge, per-thread SQLite connections |
| KV workspaces | Config, metadata, single-doc API responses | `kv_reader` with `get_keys`/`get_value` |

**You now have everything needed to build production-grade agent tools with ctxtual.**  🎉